# Extraction Dashboard

Set `run_dir` to a folder produced by `experiment_univariate.py` or `features.py`, then run the cells.

In [ ]:
from pathlib import Path
import pandas as pd
import torch

from extraction.visu.plots import (
    plot_error_distribution,
    plot_feature_scatter,
    plot_horizon_errors,
    plot_user_error_scatter,
)

run_dir = Path('../outputs/extraction_univariate/univariate')
run_dir

## Univariate Losses

In [ ]:
loss_path = run_dir / 'univariate_losses.csv'
losses = pd.read_csv(loss_path)
losses.head()

In [ ]:
plot_dir = run_dir / 'notebook_plots'
for split, df in losses.groupby('split'):
    per_user = df.groupby('user_idx')['nmse'].agg(['mean', 'std']).reset_index()
    per_user = per_user.rename(columns={'mean': 'mean_loss', 'std': 'std_loss'})
    plot_user_error_scatter(per_user, plot_dir, f'{split}_user_nmse.png', title=f'{split} per-user nMSE')
    plot_error_distribution(df['nmse'], plot_dir, f'{split}_nmse_distribution.png', title=f'{split} nMSE distribution')
plot_dir

In [ ]:
payload_path = run_dir / 'univariate_payload.pt'
if payload_path.exists():
    payload = torch.load(payload_path, map_location='cpu', weights_only=False)
    for split, item in payload.items():
        horizon = item['horizon_mse']
        if torch.is_tensor(horizon) and horizon.numel() > 0:
            plot_horizon_errors(horizon.mean(dim=(0, 1)).numpy(), plot_dir, f'{split}_horizon_mse.png')
payload_path.exists()

## Neighbor Features

In [ ]:
feature_path = run_dir / 'feature_analysis' / 'neighbor_features.csv'
if feature_path.exists():
    features = pd.read_csv(feature_path)
else:
    features = pd.DataFrame()
features.head()

In [ ]:
if not features.empty:
    plot_feature_scatter(features, 'distance_mean', 'improvement_pct', plot_dir, 'improvement_vs_distance.png')
    plot_feature_scatter(features, 'base_loss', 'context_loss', plot_dir, 'context_vs_base_loss.png')
plot_dir